# Phase 7 — GRPO fine-tune of Qwen2.5-3B on TokenEfficiencyEnv

Trains the model to answer questions correctly **with fewer tokens** by giving it our 6-component reward (correctness 55% + efficiency 15% + self-assessment 15% + redundancy/keyword/format 5% each), and lets GRPO do the rest.

**Reading order**: each cell has a one-line purpose comment up top. Run them in order. The whole notebook runs end-to-end on a single 16GB+ GPU in ~1–2h with the default 300-step config; bump `max_steps` once you trust it.

**What you'll get out**: an `outputs/grpo_qwen2.5_3b/` directory with the LoRA adapter + a `metrics.json`, plus before/after plots showing token usage going down without correctness dropping.

**What you need set**:
* `HF_TOKEN` env var (for the judge — used both during training and eval).
* `pip install -r ../requirements-train.txt` from the repo root.
* CUDA-capable GPU recommended; 16GB enough with LoRA + bf16.

In [ ]:
# 0) Colab bootstrap — fully self-contained. Runs ONLY on Google Colab.
#    On a local machine this cell is a no-op and your existing editable
#    install is used.
#
#    On Colab this cell:
#      * git-clones this repo into /content (if not already there),
#      * chdir + sys.path so everything below imports cleanly,
#      * pip-installs the env package + requirements-train.txt + unsloth
#        BEFORE any `import training` (training/__init__.py eagerly imports
#        token_efficiency_env, which needs openenv to be installed first),
#      * securely prompts for HF_TOKEN via getpass (never written to disk).
#
#    The first run on Colab takes ~3 min to install; subsequent cells run
#    against the cloned repo. If you forked the repo, change REPO_URL.
REPO_URL = "https://github.com/SaishAmbar/token-efficiency-env.git"

import os, subprocess, sys

_IS_COLAB = "google.colab" in sys.modules

if _IS_COLAB:
    REPO_DIR = "/content/token-efficiency-env"

    if not os.path.exists(REPO_DIR):
        print(f">>> Cloning {REPO_URL} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True,
        )
    else:
        print(f">>> Repo already present at {REPO_DIR}")

    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Install order MATTERS: env package first (brings in openenv), then
    # training deps, then unsloth. Do NOT `import training` before this.
    print(">>> Installing token_efficiency_env (editable) ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", "./token_efficiency_env"],
        check=True,
    )
    print(">>> Installing requirements-train.txt ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"],
        check=True,
    )
    print(">>> Installing unsloth ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "unsloth"],
        check=True,
    )

    # Purge any half-loaded `training` / `token_efficiency_env` modules from
    # a prior failed attempt so later cells get a clean import graph.
    for _mod in [
        k for k in list(sys.modules.keys())
        if k == "training" or k.startswith("training.")
           or k == "token_efficiency_env" or k.startswith("token_efficiency_env.")
    ]:
        del sys.modules[_mod]

    # Secure HF_TOKEN prompt (judge API, never written to disk)
    if not os.environ.get("HF_TOKEN"):
        import getpass
        _tok = getpass.getpass(
            "Paste your HF_TOKEN (read access is enough). Press Enter to skip: "
        )
        if _tok.strip():
            os.environ["HF_TOKEN"] = _tok.strip()
            print("HF_TOKEN set.")
        else:
            print("No token — judge will fall back to keyword scoring (fine for smoke).")
    else:
        print("HF_TOKEN already set.")

    print(f"\n>>> Bootstrap ready. cwd = {os.getcwd()}")
else:
    print("Local kernel detected; skipping Colab bootstrap (using existing install).")

## Colab playbook (run this in order)

**Step 0 — set runtime.** `Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save`. Free tier is enough for smoke + full runs.

**Step 1 — run cells top-to-bottom.** No manual `pip install` needed — the bootstrap cell above handles it.

**Step 2 — smoke run first.** The config cell below has `SMOKE = True` by default. This is a ~15-20 min run with:
- 50 GRPO steps (vs 300 in the full run)
- keyword judge (no HF API — works with or without `HF_TOKEN`)
- 24-prompt starter bank (skips the 2.3k loader download)
- Unsloth + SFT warmup **on** (the very things smoke mode is meant to verify)

**Step 3 — check the smoke report.** The second-to-last cell writes `training/smoke_report.json` and prints a one-screen summary. Copy-paste that summary back to the chat so I can green-light the full run.

**What a healthy smoke looks like:**
- `cliff_rate` drops (baseline ~60-100% → trained <30%) — SFT warmup did its job
- `mean_reward` goes up (even if still negative — direction matters more than magnitude on 50 steps)
- Training loss trends down in the plot

**Step 4 — full run.** Once smoke passes, flip `SMOKE = False` and re-run from the config cell downward. The full run uses the 2.3k-prompt bank + real HF judge and takes ~1-2h on T4.

In [ ]:
# 1) Path setup — make `training` and `token_efficiency_env` importable when
#    you launch jupyter from anywhere.
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT =", REPO_ROOT)
print("HF_TOKEN  =", "set" if os.environ.get("HF_TOKEN") else "NOT SET — judge will fall back to keyword scoring")

In [ ]:
# 2) Config — single source of truth. Edit values here, NOT in cells below.
#
# ┌─────────────────────────────────────────────────────────────────────┐
# │  SMOKE MODE — run this FIRST, before the full 300-step training.    │
# │                                                                     │
# │   SMOKE=True  → 50 steps, keyword judge, starter bank, ~15-20 min.  │
# │                 Goal: prove the pipeline runs end-to-end, no OOM,   │
# │                 no TRL drift, no dep errors.                        │
# │   SMOKE=False → 300 steps, real HF judge, 2.3k-prompt full bank.    │
# │                 Only flip this AFTER smoke has passed.              │
# └─────────────────────────────────────────────────────────────────────┘
SMOKE = True  # ← flip to False for the real Step-6 run

from training import TrainingConfig, is_colab

if SMOKE:
    config = TrainingConfig.smoke(colab=is_colab())
else:
    # Colab auto-enables the accelerated + large-bank + SFT-warmup path.
    # Locally (no GPU) all three flip off so tests / Windows devs keep working.
    config = TrainingConfig(
        use_unsloth=is_colab(),
        prompt_bank_mode="full" if is_colab() else "starter",
        use_sft_warmup=is_colab(),
    )

print(config.describe())

In [ ]:
# 3) Train/holdout split — print so you can SEE what the model never saw.
#    Honours `config.prompt_bank_mode`: on Colab this builds the split over
#    the ~2.3k programmatic bank; locally it uses the 24-prompt starter.
from training import describe_split, holdout_prompts, train_prompts

print(describe_split(mode=config.prompt_bank_mode))
print(
    f"\nTraining on {len(train_prompts(config.prompt_bank_mode))} prompts, "
    f"evaluating on {len(holdout_prompts(config.prompt_bank_mode))}."
)

## 4) (Optional) Server pool

Skip this cell if `config.reward_backend == "in_process"` (the default). Only run it if you flipped to `"ws"` to stress-test the deployed server path.

In [ ]:
# 4) Server pool — only spawned if reward_backend == "ws".
pool = None
if config.reward_backend == "ws":
    from training import ServerPool
    os.environ["JUDGE_BACKEND"] = config.judge_backend  # workers inherit this
    pool = ServerPool(
        num_envs=config.num_env_servers,
        base_port=config.base_port,
        judge_backend=config.judge_backend,
    ).start()
    print("Pool ready:", pool.urls)
else:
    print("reward_backend = in_process; skipping server pool.")

In [ ]:
# 5) Reward function — wires the env into TRL's reward_funcs API.
#    The same JUDGE_BACKEND env var is honoured by the in-process env.
from training import RewardLog, build_reward_func

os.environ["JUDGE_BACKEND"] = config.judge_backend
reward_log = RewardLog()
reward_func, reward_adapter = build_reward_func(config, log=reward_log)

_test_rewards = reward_func(
    ["What is the capital of France?"] * 2,
    ["<budget>3</budget><answer>Paris.</answer>",
     "<budget>3</budget><answer>London.</answer>"],
)
print("smoke-test rewards (Paris vs London):", _test_rewards)
assert _test_rewards[0] > _test_rewards[1], "sanity check: Paris should out-score London"

In [ ]:
# 6) Tokenizer + base model + LoRA.
#
# Two paths, toggled by `config.use_unsloth`:
#   • Unsloth (Linux + CUDA): ~2× training throughput and ~40% lower VRAM
#     per the Unsloth/TRL cookbook; loads in 4-bit by default.
#   • Vanilla HF + PEFT (default): runs anywhere torch does, bf16 on
#     recent GPUs or fp16 fallback. This is the tested baseline.
#
# Flip on Unsloth with:   TrainingConfig(use_unsloth=True)
# Requires `pip install unsloth` on a Linux + NVIDIA box (WSL2 OK).
import torch

if config.use_unsloth:
    # Unsloth's FastLanguageModel wraps HF + its own kernels. It returns
    # its own tokenizer that pads correctly for LoRA + 4-bit out of the box.
    from unsloth import FastLanguageModel  # type: ignore[import-not-found]

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.model_name,
        max_seq_length=config.unsloth_max_seq_length,
        load_in_4bit=config.unsloth_load_in_4bit,
        dtype=None,  # let Unsloth pick bf16 / fp16 based on the GPU
        trust_remote_code=True,
    )
    dtype = model.dtype if hasattr(model, "dtype") else torch.bfloat16
    print(
        f"Model loaded via Unsloth | device={model.device} | dtype={dtype} | "
        f"4bit={config.unsloth_load_in_4bit}"
    )

    if config.use_lora:
        model = FastLanguageModel.get_peft_model(
            model,
            r=config.lora_r,
            lora_alpha=config.lora_alpha,
            lora_dropout=config.lora_dropout,
            target_modules=config.lora_target_modules,
            bias="none",
            use_gradient_checkpointing="unsloth",  # Unsloth's optimized path
            random_state=config.seed,
        )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(
        config.model_name, trust_remote_code=True
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    print(f"Model loaded via HF | device={model.device} | dtype={dtype}")

    if config.use_lora:
        from peft import LoraConfig, get_peft_model
        lora_cfg = LoraConfig(
            r=config.lora_r,
            lora_alpha=config.lora_alpha,
            lora_dropout=config.lora_dropout,
            target_modules=config.lora_target_modules,
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora_cfg)

if config.use_lora and hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()

In [ ]:
# 6.5) SFT format warmup — teaches Qwen the <budget>N</budget><answer>...</answer>
#      shape in ~2 min on a T4, so GRPO gets a real reward signal from step 1.
#      Skip by setting config.use_sft_warmup=False (default off locally; on for Colab).
if config.use_sft_warmup:
    from datasets import Dataset as _HFDataset
    from trl import SFTConfig, SFTTrainer
    from training import build_format_teaching_examples
    from notebooks.eval_baseline_vs_trained import SYSTEM_PROMPT as _SFT_SYSTEM_PROMPT

    def _tok_len(s: str) -> int:
        return len(tokenizer.encode(s))

    sft_examples = build_format_teaching_examples(
        n=config.sft_warmup_examples, tokenize_len=_tok_len, seed=config.seed
    )

    sft_rows = []
    for ex in sft_examples:
        text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": _SFT_SYSTEM_PROMPT},
                {"role": "user",   "content": ex["question"]},
                {"role": "assistant", "content": ex["response"]},
            ],
            tokenize=False,
            add_generation_prompt=False,
        )
        sft_rows.append({"text": text})

    sft_ds = _HFDataset.from_list(sft_rows)
    print(f"SFT warmup dataset: {len(sft_ds)} examples. First row preview:")
    print(sft_ds[0]["text"][:400], "...")

    sft_cfg = SFTConfig(
        output_dir=f"{config.output_dir}/sft_warmup",
        num_train_epochs=config.sft_warmup_epochs,
        per_device_train_batch_size=config.sft_warmup_batch_size,
        learning_rate=config.sft_warmup_lr,
        logging_steps=5,
        save_strategy="no",
        bf16=(dtype == torch.bfloat16),
        fp16=(dtype == torch.float16),
        report_to=[],
        seed=config.seed,
    )
    sft_trainer = SFTTrainer(
        model=model,
        args=sft_cfg,
        train_dataset=sft_ds,
        processing_class=tokenizer,
    )
    sft_trainer.train()
    print("\nSFT warmup complete — model should now emit valid <budget>/<answer> format.")
else:
    print("SFT warmup disabled (config.use_sft_warmup=False). GRPO will start cold.")

# 7) Build the training dataset. GRPO eats {"prompt": str} rows.
#    The system prompt is identical to the one in eval_baseline_vs_trained.py
#    so before/after numbers stay comparable.
from datasets import Dataset
from notebooks.eval_baseline_vs_trained import SYSTEM_PROMPT

def _to_chat(question: str) -> str:
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

train_ds = Dataset.from_list([
    {"prompt": _to_chat(t["prompt"]), "raw_prompt": t["prompt"]}
    for t in train_prompts(config.prompt_bank_mode)
])
print(train_ds)
print("\nFirst rendered prompt:\n", train_ds[0]["prompt"][:400], "\n...")

In [ ]:
# 8) Reward-function shim that maps TRL's CHAT-rendered prompt back to the
#    raw question, so the env's prompt → task lookup still works.
_chat_to_raw = {row["prompt"]: row["raw_prompt"] for row in train_ds}

def trl_reward(prompts, completions, **kw):
    raw_prompts = [_chat_to_raw.get(p, p) for p in prompts]
    return reward_func(raw_prompts, completions, **kw)

In [ ]:
# 9) Baseline eval BEFORE training — establishes the bar to beat.
from notebooks.eval_baseline_vs_trained import evaluate, print_summary

baseline = evaluate(
    model, tokenizer, holdout_prompts(config.prompt_bank_mode),
    label="baseline (untrained Qwen2.5-3B)",
    max_new_tokens=config.max_completion_length,
    temperature=config.eval_temperature,
    samples_per_prompt=config.eval_samples_per_prompt,
)
print_summary(baseline)

In [ ]:
# 10) GRPO trainer setup. Most knobs come straight from `config`.
from trl import GRPOConfig, GRPOTrainer

grpo_cfg = GRPOConfig(
    output_dir=config.output_dir,
    num_train_epochs=1,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    num_generations=config.num_generations,
    max_prompt_length=config.max_prompt_length,
    max_completion_length=config.max_completion_length,
    temperature=config.temperature,
    seed=config.seed,
    logging_steps=5,
    save_steps=max(50, config.max_steps // 6),
    bf16=(dtype == torch.bfloat16),
    fp16=(dtype == torch.float16),
    report_to=[],  # no wandb/tensorboard by default; flip on if you want
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_cfg,
    train_dataset=train_ds,
    reward_funcs=[trl_reward],
)
print("GRPOTrainer ready.")

In [ ]:
# 11) TRAIN. This is the long-running cell. Reward log is filled as it goes.
trainer.train()
print("\nTraining done.\n")
trainer.save_model(config.output_dir)
print(f"Checkpoint saved to: {config.output_dir}")

In [ ]:
# 12) Trained-model eval — same held-out prompts, same harness.
trained = evaluate(
    model, tokenizer, holdout_prompts(config.prompt_bank_mode),
    label="trained (LoRA adapter applied)",
    max_new_tokens=config.max_completion_length,
    temperature=config.eval_temperature,
    samples_per_prompt=config.eval_samples_per_prompt,
)
print_summary(trained)

In [ ]:
# 13) Side-by-side comparison + plots.
import json
import pandas as pd
import matplotlib.pyplot as plt
from notebooks.eval_baseline_vs_trained import summary_to_dict

comp = pd.DataFrame({
    "metric": ["reward", "correctness", "tokens_used", "overshoot_rate", "cliff_rate"],
    "baseline": [baseline.mean_reward, baseline.mean_correctness,
                 baseline.mean_tokens_used, baseline.overshoot_rate,
                 baseline.cliff_rate],
    "trained":  [trained.mean_reward, trained.mean_correctness,
                 trained.mean_tokens_used, trained.overshoot_rate,
                 trained.cliff_rate],
})
comp["delta"] = comp["trained"] - comp["baseline"]
display(comp)

rewards_df = pd.DataFrame(reward_log.to_records())
if not rewards_df.empty:
    rewards_df["step"] = rewards_df.index // max(config.num_generations, 1)
    rolling = rewards_df.groupby("step")["reward"].mean().rolling(25, min_periods=1).mean()
    plt.figure(figsize=(10, 4))
    plt.plot(rolling.index, rolling.values)
    plt.xlabel("GRPO step")
    plt.ylabel("reward (rolling mean over 25 steps)")
    plt.title("Training-time reward")
    plt.grid(True, alpha=0.3)
    plt.show()

metrics = {
    "config": config.describe(),
    "baseline": summary_to_dict(baseline),
    "trained":  summary_to_dict(trained),
}
Path(config.output_dir).mkdir(parents=True, exist_ok=True)
metrics_path = Path(config.output_dir) / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"Wrote {metrics_path}")

# Judge-ready artifacts ------------------------------------------------
# Writes before_after_report.md (prose + tables + 3 sample prompts),
# before_after_plot.png (5-metric bar chart) and reward_curve.png
# (training-time reward). Everything a hackathon judge can read on
# GitHub without running the notebook.
from training.compare_report import (
    plot_before_after,
    plot_reward_curve,
    render_markdown_report,
)

report_dir = Path("training")
report_dir.mkdir(parents=True, exist_ok=True)

md_path = render_markdown_report(
    baseline, trained,
    output_path=report_dir / "before_after_report.md",
    run_label="smoke" if SMOKE else "full",
    config_banner=config.describe(),
)
plot_path = plot_before_after(
    baseline, trained,
    output_path=report_dir / "before_after_plot.png",
)
curve_path = plot_reward_curve(
    reward_log,
    output_path=report_dir / "reward_curve.png",
    num_generations=config.num_generations,
)
print(f"Wrote {md_path}")
print(f"Wrote {plot_path}")
if curve_path is not None:
    print(f"Wrote {curve_path}")

In [ ]:
# 13.5) Smoke-run receipt — writes training/smoke_report.json with the
#       numbers that decide whether to proceed to the full run.
#
#       What "PASS" means for a smoke run:
#       * Training completed without OOM / TRL API errors.
#       * cliff_rate dropped (SFT + GRPO taught valid format).
#       * mean_reward increased (model actually learned something).
#
#       Copy this JSON back to the chat so I can verify Step 5 passed
#       before we spend 1-2h on the full run in Step 6.
import json, time
from pathlib import Path

def _fmt_delta(d: float, *, higher_is_better: bool = True) -> str:
    arrow = "▲" if (d > 0) == higher_is_better else ("▼" if d != 0 else "–")
    return f"{arrow} {d:+.4f}"

smoke_report = {
    "mode": "smoke" if SMOKE else "full",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "is_colab": is_colab(),
    "config_banner": config.describe(),
    "baseline": {
        "mean_reward":       baseline.mean_reward,
        "mean_correctness":  baseline.mean_correctness,
        "mean_tokens_used":  baseline.mean_tokens_used,
        "cliff_rate":        baseline.cliff_rate,
        "overshoot_rate":    baseline.overshoot_rate,
    },
    "trained": {
        "mean_reward":       trained.mean_reward,
        "mean_correctness":  trained.mean_correctness,
        "mean_tokens_used":  trained.mean_tokens_used,
        "cliff_rate":        trained.cliff_rate,
        "overshoot_rate":    trained.overshoot_rate,
    },
    "deltas": {
        "mean_reward":       trained.mean_reward      - baseline.mean_reward,
        "mean_correctness":  trained.mean_correctness - baseline.mean_correctness,
        "mean_tokens_used":  trained.mean_tokens_used - baseline.mean_tokens_used,
        "cliff_rate":        trained.cliff_rate       - baseline.cliff_rate,
    },
    "verdict": {
        "reward_improved":   trained.mean_reward > baseline.mean_reward,
        "format_improved":   trained.cliff_rate < baseline.cliff_rate,
        "tokens_reduced":    trained.mean_tokens_used < baseline.mean_tokens_used,
    },
}
passed = (
    smoke_report["verdict"]["reward_improved"]
    and smoke_report["verdict"]["format_improved"]
)
smoke_report["smoke_passed"] = passed

report_path = Path("training/smoke_report.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(smoke_report, indent=2))

print("=" * 64)
print(f"SMOKE REPORT  ({'smoke' if SMOKE else 'full'} mode)")
print("=" * 64)
print(f"  reward     : {baseline.mean_reward:+.4f} → {trained.mean_reward:+.4f}   "
      f"({_fmt_delta(smoke_report['deltas']['mean_reward'])})")
print(f"  cliff rate : {baseline.cliff_rate:.1%} → {trained.cliff_rate:.1%}   "
      f"({_fmt_delta(smoke_report['deltas']['cliff_rate'], higher_is_better=False)})")
print(f"  tokens     : {baseline.mean_tokens_used:.1f} → {trained.mean_tokens_used:.1f}   "
      f"({_fmt_delta(smoke_report['deltas']['mean_tokens_used'], higher_is_better=False)})")
print("-" * 64)
print(f"  SMOKE PASSED: {passed}")
print(f"  report written to: {report_path.resolve()}")
print("=" * 64)

In [ ]:
# 14) Cleanup — close adapter (no-op for in_process) and shut down the
#     server pool if we spawned one. Wrap in try so a crash above still
#     releases ports.
try:
    reward_adapter.close()
finally:
    if pool is not None:
        pool.shutdown()
        print("Server pool shut down.")
    else:
        print("Done.")